In [1]:
import os
import pandas as pd
import geopandas as gpd

# Station Metadata

In [2]:
stations_metadata = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt"
stations_df = pd.read_fwf(stations_metadata, header=None)
stations_df = stations_df.iloc[:, :5]
stations_df.columns = ["SITE_ID", "LATITUDE", "LONGITUDE", "ELEVATION", "SITE_NAME"]
stations_df.head()

,SITE_ID,LATITUDE,LONGITUDE,ELEVATION,SITE_NAME
0,ACW00011604,17.1167,-61.7833,10.1,ST JOHNS COOLIDGE FLD
1,ACW00011647,17.1333,-61.7833,19.2,ST JOHNS
2,AE000041196,25.3330,55.5170,34.0,SHARJAH INTER. AIRP
3,AEM00041194,25.2550,55.3640,10.4,DUBAI INTL
4,AEM00041217,24.4330,54.6510,26.8,ABU DHABI INTL


In [3]:
stations_df.shape

(129657, 5)

In [4]:
stations_df["SITE_ID"] = stations_df["SITE_ID"].astype(str)
stations_df["LATITUDE"] = stations_df["LATITUDE"].astype(float)
stations_df["LONGITUDE"] = stations_df["LONGITUDE"].astype(float)
stations_df["ELEVATION"] = stations_df["ELEVATION"].astype(float)
stations_df["SITE_NAME"] = stations_df["SITE_NAME"].astype(str)
stations_df["geometry"] = gpd.points_from_xy(stations_df["LONGITUDE"], stations_df["LATITUDE"], crs="EPSG:4326")
stations_gdf = gpd.GeoDataFrame(stations_df, geometry="geometry")
stations_gdf.head()

,SITE_ID,LATITUDE,LONGITUDE,ELEVATION,SITE_NAME,geometry
0,ACW00011604,17.1167,-61.7833,10.1,ST JOHNS COOLIDGE FLD,POINT (-61.7833 17.1167)
1,ACW00011647,17.1333,-61.7833,19.2,ST JOHNS,POINT (-61.7833 17.1333)
2,AE000041196,25.3330,55.5170,34.0,SHARJAH INTER. AIRP,POINT (55.517 25.333)
3,AEM00041194,25.2550,55.3640,10.4,DUBAI INTL,POINT (55.364 25.255)
4,AEM00041217,24.4330,54.6510,26.8,ABU DHABI INTL,POINT (54.651 24.433)


In [5]:
stations_gdf.to_parquet("/workspaces/stormhub/data/0_source/ghcnd/ghcnd_stations.parquet", index=False)

# Database Setup

In [6]:
test_file = "/workspaces/stormhub/data/0_source/NSF_CausesData/USC00010140.csv"
df = pd.read_csv(test_file)
df.head()

,station,region,state,date,season,precip,qflag,etc_dist,etc_min_slp,etc_avg_slp,...,front_type,ar_dist,ar_ivt,ar_iwv,ar_u850,ar_v850,pw,omega,cause_1,cause_2
0,USC00010140,6,AL,19490101,W,0.0,,1560,985.9,989.8,...,88888,2115,271,26,3.7,2.7,6.2,0.16,NONE,NONE
1,USC00010140,6,AL,19490102,W,0.0,,1294,1006.3,1006.3,...,88888,2115,277,23,5.3,0.8,24.1,-0.01,NONE,NONE
2,USC00010140,6,AL,19490103,W,0.0,,1126,991.8,997.2,...,88888,99999,99999,99999,99999.0,99999.0,36.0,-0.46,NONE,NONE
3,USC00010140,6,AL,19490104,W,16.0,,1013,996.4,1002.3,...,88888,99999,99999,99999,99999.0,99999.0,37.7,-0.36,NONE,NONE
4,USC00010140,6,AL,19490105,W,33.3,,1254,1007.0,1007.0,...,88888,4163,362,16,9.3,16.2,45.6,-0.42,NONE,NONE


In [7]:
folder_path = "/workspaces/stormhub/data/0_source/NSF_CausesData/"
file_list = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

In [8]:
import pyarrow as pa
import pyarrow.parquet as pq

def csvs_to_parquet_stream(
    file_list,
    folder_path,
    output_parquet,
    chunksize=25_000,
    csv_kwargs=None,
    verbose=True,
    date_cols=("date", "DATE"),
    ):
    """
    Stream CSV files into a single Parquet file without loading all data into memory.

    Args:
        file_list (list[str]): CSV filenames (not full paths).
        folder_path (str): Directory containing the CSVs.
        output_parquet (str): Output Parquet file path.
        chunksize (int|None): Number of rows per chunk to read. Use None to read whole file.
        csv_kwargs (dict|None): Extra kwargs for pandas.read_csv.
        verbose (bool): Print progress.
        date_cols (tuple[str]|list[str]): Column names to parse as datetime.
    """
    csv_kwargs = dict(csv_kwargs or {})
    csv_kwargs.setdefault("na_values", ["NONE"])
    csv_kwargs.setdefault("keep_default_na", True)
    writer = None
    total_rows = 0
    base_schema = None
    base_columns = None
    numeric_cols = None
    try:
        for i, fname in enumerate(file_list, start=1):
            csv_path = os.path.join(folder_path, fname)
            if verbose:
                print(f"[{i}/{len(file_list)}] Reading {csv_path}")
            if chunksize is None:
                chunk_iter = [pd.read_csv(csv_path, **csv_kwargs)]
            else:
                chunk_iter = pd.read_csv(csv_path, chunksize=chunksize, **csv_kwargs)
            for chunk in chunk_iter:
                if hasattr(chunk, "columns"):
                    chunk.columns = chunk.columns.str.strip()
                if date_cols:
                    for col in date_cols:
                        if col in chunk.columns:
                            chunk[col] = pd.to_datetime(chunk[col], errors="coerce")
                if base_columns is None:
                    base_columns = list(chunk.columns)
                    table = pa.Table.from_pandas(chunk, preserve_index=False)
                    base_schema = table.schema
                    numeric_cols = [
                        field.name
                        for field in base_schema
                        if pa.types.is_integer(field.type) or pa.types.is_floating(field.type)
                    ]
                    writer = pq.ParquetWriter(output_parquet, base_schema)
                else:
                    chunk = chunk.reindex(columns=base_columns)
                    if numeric_cols:
                        for col in numeric_cols:
                            if col in chunk.columns:
                                chunk[col] = pd.to_numeric(chunk[col], errors="coerce")
                    table = pa.Table.from_pandas(
                        chunk,
                        schema=base_schema,
                        preserve_index=False,
                        safe=False,
                    )
                writer.write_table(table)
                total_rows += len(chunk)
        if verbose:
            print(f"Wrote {total_rows:,} rows to {output_parquet}")
    finally:
        if writer is not None:
            writer.close()

# Example usage:
csvs_to_parquet_stream(file_list, folder_path, "/workspaces/stormhub/data/0_source/ghcnd/all_stations.parquet", chunksize=25_000, date_cols=("date",))

[1/3155] Reading /workspaces/stormhub/data/0_source/NSF_CausesData/USC00010140.csv
[2/3155] Reading /workspaces/stormhub/data/0_source/NSF_CausesData/USC00010252.csv
[3/3155] Reading /workspaces/stormhub/data/0_source/NSF_CausesData/USC00010583.csv
[4/3155] Reading /workspaces/stormhub/data/0_source/NSF_CausesData/USC00010655.csv
[5/3155] Reading /workspaces/stormhub/data/0_source/NSF_CausesData/USC00011084.csv
[6/3155] Reading /workspaces/stormhub/data/0_source/NSF_CausesData/USC00011566.csv
[7/3155] Reading /workspaces/stormhub/data/0_source/NSF_CausesData/USC00011694.csv
[8/3155] Reading /workspaces/stormhub/data/0_source/NSF_CausesData/USC00011725.csv
[9/3155] Reading /workspaces/stormhub/data/0_source/NSF_CausesData/USC00012245.csv
[10/3155] Reading /workspaces/stormhub/data/0_source/NSF_CausesData/USC00012813.csv
[11/3155] Reading /workspaces/stormhub/data/0_source/NSF_CausesData/USC00013160.csv
[12/3155] Reading /workspaces/stormhub/data/0_source/NSF_CausesData/USC00013511.csv
[